In [ ]:
!pip -q install --upgrade pip
!pip -q install "dwave-ocean-sdk==6.9.0"
!pip -q install scikit-learn pandas numpy scipy matplotlib

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('/content/survey lung cancer.csv')

In [ ]:
dfen2 = df.copy()
dfen2['GENDER'] = dfen2['GENDER'].map({'F':0,'M':1})
dfen2['LUNG_CANCER'] = dfen2['LUNG_CANCER'].map({'NO':0,'YES':1})

In [ ]:
# ------------------------------------------
#  Install imblearn once (if not installed)
# ------------------------------------------
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# ------------------------------------------
# 1️⃣  Verify dataset & identify target column
# ------------------------------------------
# You already have dfs
print("Original shape:", dfen2.shape)
print("Original class distribution:")
print(dfen2['LUNG_CANCER'].value_counts())

# ------------------------------------------
# 2️⃣  Separate features (X) and target (y)
# ------------------------------------------
X = dfen2.drop(columns=['LUNG_CANCER']).values
y = dfen2['LUNG_CANCER'].values

# ------------------------------------------
# 3️⃣  Apply SMOTE only on the minority class
# ------------------------------------------
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ------------------------------------------
# 4️⃣  Rebuild into a balanced DataFrame
# ------------------------------------------
columns = dfen2.drop(columns=['LUNG_CANCER']).columns
dfs_smotes = pd.DataFrame(X_res, columns=columns)
dfs_smotes['LUNG_CANCER'] = y_res


# ------------------------------------------
# 5️⃣  Check the new class balance
# ------------------------------------------
print("\nAfter SMOTE:")
print(dfs_smotes['LUNG_CANCER'].value_counts())
print("New shape:", dfs_smotes.shape)

In [ ]:
dff=dfs_smotes[:]
dff

In [ ]:
dff.info()

In [ ]:
dff.info()

**<h1>Asvm**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)

import dimod
from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

# ============================================================
# USER CONTROLS
# ============================================================
SPLIT_MODE = "holdout"
N_RUNS = 1
TEST_SIZE = 0.8
NUM_READS_MODEL = 6000
RANDOM_SEED_BASE = 42
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "live_qsvm_results.csv"
n_splits = 65  # repeated random subsampling splits

# ============================================================
# LOAD DATA (dff must exist)
# ============================================================
X_df = dff.drop(columns=[TARGET])
feature_names = X_df.columns.tolist()
y = dff[TARGET].astype(int).values
X_raw = X_df.values.astype(float)
print(dff.info())

# ============================================================
# SELECTED FEATURES (user-specified)
# ============================================================
selected_features_indices = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]  # exact sequence
selected_feature_names = [feature_names[i] for i in selected_features_indices]
X_raw = X_raw[:, selected_features_indices]

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

def solve_bqm(bqm, reads):
    sampler = SimulatedAnnealingSampler()
    return sampler.sample(bqm, num_reads=reads).first.sample

# ============================================================
# DUAL QSVM
# ============================================================
def build_dual_qsvm(X, y, n_bits=3, C=1.0, gamma_penalty=5.0, kernel="linear", rbf_gamma=1.0):
    y = (2*y - 1).astype(float)
    n = X.shape[0]
    if kernel == "linear":
        K = X @ X.T
    else:
        sq = np.sum(X**2, axis=1, keepdims=True)
        dists = sq + sq.T - 2*(X @ X.T)
        K = np.exp(-rbf_gamma*dists)
    scale = C/(2**n_bits - 1)
    bqm = BinaryQuadraticModel(vartype=dimod.BINARY)
    def name(i, k): return f"a_{i}_{k}"
    for i in range(n):
        for k in range(n_bits):
            coeff = (2**k)*scale
            bqm.add_variable(name(i, k), -coeff)
    for i in range(n):
        for j in range(i, n):
            for ki in range(n_bits):
                for kj in range(n_bits):
                    ci = (2**ki)*scale
                    cj = (2**kj)*scale
                    quad = 0.5*y[i]*y[j]*K[i,j]*ci*cj
                    quad += gamma_penalty*y[i]*y[j]*ci*cj
                    if i == j and ki == kj:
                        bqm.add_variable(name(i,ki), bqm.get_linear(name(i,ki)) + quad)
                    else:
                        bqm.add_interaction(name(i,ki), name(j,kj), quad)
    return bqm

def extract_dual_weights(sample, X, y, n_bits=3, C=1.0):
    y = (2*y - 1).astype(float)
    n = X.shape[0]
    scale = C/(2**n_bits - 1)
    alpha = np.zeros(n)
    for i in range(n):
        for k in range(n_bits):
            alpha[i] += (2**k)*sample.get(f"a_{i}_{k}",0)
        alpha[i] *= scale
    return (alpha*y) @ X

# ============================================================
# PRIMAL QSVM
# ============================================================
def build_squared_qsvm(X, y, n_bits=3, C=1.0, lam=0.12):
    y = (2*y - 1).astype(float)
    XtX = X.T @ X
    Xty = X.T @ y
    scale = 1/(2**n_bits - 1)
    d = X.shape[1]
    bqm = BinaryQuadraticModel(vartype=dimod.BINARY)
    def name(j,k): return f"s_{j}_{k}"
    for j in range(d):
        for k in range(n_bits):
            cj = (2**k)*scale
            lin = -2*C*cj*Xty[j] + C*(cj**2)*XtX[j,j] + lam*(cj**2)
            bqm.add_variable(name(j,k), lin)
    for i in range(d):
        for j in range(i+1,d):
            for ki in range(n_bits):
                for kj in range(n_bits):
                    ci = (2**ki)*scale
                    cj = (2**kj)*scale
                    quad = 2*C*ci*cj*XtX[i,j]
                    bqm.add_interaction(name(i,ki), name(j,kj), quad)
    return bqm

def extract_primal_weights(sample, d, n_bits=3):
    scale = 1/(2**n_bits - 1)
    w = np.zeros(d)
    for j in range(d):
        for k in range(n_bits):
            w[j] += (2**k)*sample.get(f"s_{j}_{k}",0)
        w[j] *= scale
    return w

def predict(X, w):
    return 1/(1+np.exp(-(X@w)))

# ============================================================
# MAIN LOOP WITH REPEATED RANDOM SUBSAMPLING (n_splits)
# ============================================================
summary = []
total_models = 2  # only selected models
completed_jobs = 0

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, n_splits + 1):

    # Repeated random subsampling
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * RANDOM_SEED_BASE
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    # Model configurations
    dual_configs = {
        "DW_QSVM_SoftMargin_LowC": dict(kernel="linear",C=0.3,gamma_penalty=5),
    }

    primal_configs = {
        #"DW_QSVM_SquaredHinge": dict(C=1,lam=0.12,n_bits=3),
    }

    for name, cfg in {**dual_configs, **primal_configs}.items():

        start = time.time()

        if name in dual_configs:
            bqm = build_dual_qsvm(X_train, y_train, **cfg)
            sample = solve_bqm(bqm, NUM_READS_MODEL)
            w = extract_dual_weights(sample, X_train, y_train, C=cfg["C"])
        else:
            bqm = build_squared_qsvm(
                X_train, y_train,
                n_bits=cfg["n_bits"],
                C=cfg["C"],
                lam=cfg["lam"]
            )
            sample = solve_bqm(bqm, NUM_READS_MODEL)
            w = extract_primal_weights(sample, X_train.shape[1], n_bits=cfg["n_bits"])

        metrics = compute_metrics(y_test, predict(X_test, w))
        runtime = time.time() - start
        completed_jobs += 1
        progress = (completed_jobs/(total_models*n_splits))*100

        print(
            f"[{progress:6.2f}%] "
            f"{name} | "
            f"Acc={metrics['Accuracy']:.4f} | "
            f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
            f"F1={metrics['F1']:.4f} | "
            f"Precision={metrics['Precision']:.4f} | "
            f"Sensitivity={metrics['Sensitivity']:.4f} | "
            f"Specificity={metrics['Specificity']:.4f} | "
            f"Kappa={metrics['Kappa']:.4f} | "
            f"Time={runtime:.2f}s"
        )

        row = {
            "Model": name,
            "Accuracy_mean": metrics["Accuracy"],
            "ROC_AUC_mean": metrics["ROC-AUC"],
            "F1_mean": metrics["F1"],
            "Precision": metrics["Precision"],
            "Sensitivity": metrics["Sensitivity"],
            "Specificity": metrics["Specificity"],
            "Kappa": metrics["Kappa"],
            "Runtime_sec": runtime
        }

        summary.append(row)

pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary).sort_values("Accuracy_mean", ascending=False).reset_index(drop=True)
print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>For ASVM related figs**

In [ ]:
import numpy as np
import pandas as pd
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    cohen_kappa_score, confusion_matrix, roc_curve
)

import dimod
from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

# ============================================================
# USER CONTROLS
# ============================================================
SPLIT_MODE = "holdout"
N_RUNS = 1
TEST_SIZE = 0.8
NUM_READS_MODEL = 6000
RANDOM_SEED_BASE = 504
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "live_qsvm_results.csv"

# Fixed features (0-based indexing)
selected_features_indices = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

# ============================================================
# LOAD DATA (dff must exist)
# ============================================================
X_df = dff.drop(columns=[TARGET])
feature_names = X_df.columns.tolist()

y = dff[TARGET].astype(int).values
X_raw = X_df.values.astype(float)

print(dff.info())

# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================
def plot_energy_convergence(sampleset):
    energies = sampleset.record.energy
    best = np.minimum.accumulate(energies)

    plt.figure()
    plt.plot(best)
    plt.title("QUBO Energy Convergence")
    plt.xlabel("Iteration")
    plt.ylabel("Best Energy")
    plt.show()

def plot_energy_hist(sampleset):
    energies = sampleset.record.energy
    plt.figure()
    plt.hist(energies, bins=30)
    plt.title("QUBO Energy Distribution")
    plt.xlabel("Energy")
    plt.ylabel("Frequency")
    plt.show()

def plot_solution_frequency(sampleset):
    df = sampleset.to_pandas_dataframe()
    counts = df.groupby(list(sampleset.variables)).size()
    counts = counts.sort_values(ascending=False)[:10]

    plt.figure()
    counts.plot(kind="bar")
    plt.title("Annealing Solution Frequency")
    plt.ylabel("Counts")
    plt.show()

def plot_energy_landscape(sampleset):
    energies = sampleset.record.energy
    unique_e, counts = np.unique(energies, return_counts=True)

    plt.figure(figsize=(7,5))
    plt.scatter(unique_e, counts)
    plt.xlabel("Energy Level")
    plt.ylabel("Number of States")
    plt.title("Annealing Energy Landscape")
    plt.show()

def plot_qubo_heatmap(bqm):
    var_order = list(bqm.variables)
    mat = bqm.to_numpy_matrix(variable_order=var_order)

    plt.figure(figsize=(6,6))
    sns.heatmap(mat, cmap="coolwarm",
                xticklabels=var_order,
                yticklabels=var_order)
    plt.title("QUBO Matrix Heatmap")
    plt.show()

def plot_qubo_graph(bqm):
    G = nx.Graph()
    for v in bqm.variables:
        G.add_node(v)
    for (u,v),bias in bqm.quadratic.items():
        if abs(bias) > 0:
            G.add_edge(u,v,weight=bias)
    plt.figure(figsize=(7,7))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos,
            node_color="lightblue",
            edge_color="gray",
            with_labels=True,
            node_size=500,
            font_size=8)
    plt.title("QUBO Interaction Graph")
    plt.show()

def plot_kernel_matrix(X):
    K = X @ X.T
    plt.figure(figsize=(6,6))
    sns.heatmap(K, cmap="viridis")
    plt.title("Kernel Matrix Visualization")
    plt.show()

# ===================== NEW FUNCTION ==========================
def plot_roc_curve(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
    plt.plot([0,1], [0,1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()
# ============================================================

def plot_decision_boundary(X, y, w):
    if X.shape[1] > 2:
        X_vis = PCA(n_components=2).fit_transform(X)
    else:
        X_vis = X

    x_min, x_max = X_vis[:,0].min()-1, X_vis[:,0].max()+1
    y_min, y_max = X_vis[:,1].min()-1, X_vis[:,1].max()+1

    xx, yy = np.meshgrid(
        np.linspace(x_min,x_max,200),
        np.linspace(y_min,y_max,200)
    )

    grid = np.c_[xx.ravel(),yy.ravel()]

    if X.shape[1] > 2:
        pca = PCA(n_components=2).fit(X)
        grid_full = pca.inverse_transform(grid)
    else:
        grid_full = grid

    Z = 1/(1+np.exp(-(grid_full@w)))
    Z = Z.reshape(xx.shape)

    plt.figure()
    plt.contourf(xx,yy,Z,levels=50,cmap="coolwarm")
    plt.scatter(X_vis[:,0],X_vis[:,1],c=y,cmap="bwr")
    plt.title("Decision Boundary")
    plt.show()

def plot_support_vectors(X, y, w):
    margins = np.abs(X @ w)
    threshold = np.percentile(margins, 20)
    support_idx = margins <= threshold

    if X.shape[1] > 2:
        X_vis = PCA(n_components=2).fit_transform(X)
    else:
        X_vis = X

    plt.figure()
    plt.scatter(X_vis[:,0], X_vis[:,1], c=y, cmap="bwr", label="Samples")
    plt.scatter(X_vis[support_idx,0],
                X_vis[support_idx,1],
                s=120, facecolors='none',
                edgecolors='black',
                label="Support Vectors")
    plt.title("Support Vector Visualization")
    plt.legend()
    plt.show()

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Specificity": specificity,
        "Sensitivity": sensitivity,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

def solve_bqm(bqm, reads):
    sampler = SimulatedAnnealingSampler()
    sampleset = sampler.sample(bqm, num_reads=reads)

    plot_energy_convergence(sampleset)
    plot_energy_hist(sampleset)
    plot_solution_frequency(sampleset)
    plot_energy_landscape(sampleset)

    return sampleset.first.sample

# ============================================================
# DUAL QSVM
# ============================================================
def build_dual_qsvm(X, y, n_bits=3, C=1.0, gamma_penalty=5.0, kernel="linear", rbf_gamma=1.0):
    y = (2*y-1).astype(float)
    n = X.shape[0]

    if kernel == "linear":
        K = X @ X.T
    else:
        sq = np.sum(X**2, axis=1, keepdims=True)
        dists = sq + sq.T - 2*(X @ X.T)
        K = np.exp(-rbf_gamma*dists)

    plot_kernel_matrix(X)

    scale = C/(2**n_bits - 1)
    bqm = BinaryQuadraticModel(vartype=dimod.BINARY)
    def name(i,k): return f"a_{i}_{k}"

    for i in range(n):
        for k in range(n_bits):
            coeff = (2**k)*scale
            bqm.add_variable(name(i,k), -coeff)

    for i in range(n):
        for j in range(i,n):
            for ki in range(n_bits):
                for kj in range(n_bits):
                    ci = (2**ki)*scale
                    cj = (2**kj)*scale
                    quad = 0.5*y[i]*y[j]*K[i,j]*ci*cj
                    quad += gamma_penalty*y[i]*y[j]*ci*cj
                    if i==j and ki==kj:
                        bqm.add_variable(name(i,ki), bqm.get_linear(name(i,ki))+quad)
                    else:
                        bqm.add_interaction(name(i,ki), name(j,kj), quad)

    plot_qubo_heatmap(bqm)
    plot_qubo_graph(bqm)

    return bqm

def extract_dual_weights(sample, X, y, n_bits=3, C=1.0):
    y = (2*y-1).astype(float)
    n = X.shape[0]
    scale = C/(2**n_bits - 1)
    alpha = np.zeros(n)
    for i in range(n):
        for k in range(n_bits):
            alpha[i] += (2**k)*sample.get(f"a_{i}_{k}",0)
        alpha[i] *= scale
    return (alpha*y) @ X

# ============================================================
# PREDICTION
# ============================================================
def predict(X, w):
    return 1/(1+np.exp(-(X@w)))

# ============================================================
# MAIN LOOP
# ============================================================
summary = []

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for run in range(N_RUNS):
    tr, te = train_test_split(
        np.arange(len(y)),
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_train = sc.fit_transform(imp.fit_transform(X_raw[tr]))
    X_test = sc.transform(imp.transform(X_raw[te]))
    y_train, y_test = y[tr], y[te]

    selected = selected_features_indices
    print(f"\nSelected Features ({len(selected)}):")
    for i in selected:
        print(f"{i} -> {feature_names[i]}")

    X_train = X_train[:, selected]
    X_test = X_test[:, selected]

    dual_configs = {
        "DW_QSVM_SoftMargin_LowC": dict(kernel="linear", C=0.3, gamma_penalty=5),
    }

    for name, cfg in dual_configs.items():
        start = time.time()
        bqm = build_dual_qsvm(X_train, y_train, **cfg)
        sample = solve_bqm(bqm, NUM_READS_MODEL)
        w = extract_dual_weights(sample, X_train, y_train, C=cfg["C"])
        y_prob = predict(X_test, w)

        metrics = compute_metrics(y_test, y_prob)
        runtime = time.time() - start

        plot_decision_boundary(X_train, y_train, w)
        plot_support_vectors(X_train, y_train, w)

        # ================= NEW ROC CURVE =================
        plot_roc_curve(y_test, y_prob)

        print(name, metrics)
        summary.append(metrics)

summary_df = pd.DataFrame(summary)
print("\nDONE.")

**<h1>annealing knn**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
from sklearn.metrics.pairwise import linear_kernel

import dimod
from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

# ============================================================
# USER CONTROLS
# ============================================================
FIXED_FEATURES_IDX = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]  # Fixed feature sequence
TEST_SIZE = 0.8
NUM_READS_MODEL = 6000
RANDOM_SEED_BASE = 42
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "annealing_knn_fixed_features.csv"
n_splits = 10  # repeated random subsampling splits
K_NEIGHBORS = 3  # k for KNN
LAMBDA_PENALTY = 5.0  # QUBO constraint penalty

# ============================================================
# LOAD DATA (dff must exist)
# ============================================================
X_df = dff.drop(columns=[TARGET])
y = dff[TARGET].astype(int).values
X_raw = X_df.values.astype(float)

# ============================================================
# SELECT ONLY FIXED FEATURES
# ============================================================
X_raw = X_raw[:, FIXED_FEATURES_IDX]
selected_feature_names = [X_df.columns[i] for i in FIXED_FEATURES_IDX]
print("Using fixed features:", selected_feature_names)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_pred),
        "PR-AUC": average_precision_score(y_true, y_pred),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# ============================================================
# ANNEALING KNN PREDICTION
# ============================================================
def annealing_knn_predict(X_train, y_train, X_test, k=3, num_reads=6000, lam=5.0):
    sampler = SimulatedAnnealingSampler()
    y_pred = []

    for x in X_test:
        # similarity to all training points
        sim = linear_kernel(x.reshape(1,-1), X_train).flatten()
        n = len(sim)

        # QUBO: maximize total similarity of selected k neighbors
        Q = {}
        for i in range(n):
            Q[(i,i)] = -sim[i] + lam*(1 - 2*k)
            for j in range(i+1, n):
                Q[(i,j)] = 2*lam

        # solve QUBO
        bqm = BinaryQuadraticModel.from_qubo(Q)
        sample = sampler.sample(bqm, num_reads=num_reads).first.sample

        # select neighbors
        neighbors = [i for i,v in sample.items() if v==1]
        if len(neighbors) == 0:
            y_pred.append(0)
            continue

        labels = y_train[neighbors]
        pred = 1 if np.sum(labels) >= len(labels)/2 else 0
        y_pred.append(pred)

    return np.array(y_pred)

# ============================================================
# MAIN LOOP WITH REPEATED RANDOM SUBSAMPLING
# ============================================================
summary = []
completed_jobs = 0
total_models = 1  # only Annealing KNN

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, n_splits + 1):

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * RANDOM_SEED_BASE
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    start = time.time()
    y_pred = annealing_knn_predict(X_train, y_train, X_test, k=K_NEIGHBORS, num_reads=NUM_READS_MODEL, lam=LAMBDA_PENALTY)
    runtime = time.time() - start

    metrics = compute_metrics(y_test, y_pred)
    completed_jobs += 1
    progress = (completed_jobs / (total_models*n_splits)) * 100

    print(
        f"[{progress:6.2f}%] "
        f"Annealing_KNN | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split": split,
        "Accuracy_mean": metrics["Accuracy"],
        "ROC_AUC_mean": metrics["ROC-AUC"],
        "F1_mean": metrics["F1"],
        "Precision": metrics["Precision"],
        "Sensitivity": metrics["Sensitivity"],
        "Specificity": metrics["Specificity"],
        "Kappa": metrics["Kappa"],
        "Runtime_sec": runtime
    }

    summary.append(row)
    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary).sort_values("Accuracy_mean", ascending=False).reset_index(drop=True)
print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>AQboost**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import time
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
from sklearn.tree import DecisionTreeClassifier

import dimod
from neal import SimulatedAnnealingSampler

# ============================================================
# USER CONTROLS
# ============================================================
FIXED_FEATURES_IDX = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]
TEST_SIZE = 0.8
NUM_READS_MODEL = 5000
RANDOM_SEED_BASE = 42
TARGET = "LUNG_CANCER"
AUTO_SAVE_PATH = "qboost_fixed_features_results.csv"
n_splits = 65   # number of repeated random splits

# QBoost tuning parameters
N_WEAK = 20        # number of weak learners
LAMBDA = 0.1       # regularization in QUBO energy
K_ENS = 10         # threshold of positive votes for final class

# ============================================================
# LOAD DATA (dff must exist)
# ============================================================
X_df = dff.drop(columns=[TARGET])
y = dff[TARGET].astype(int).values
X_raw = X_df.values.astype(float)

# ============================================================
# SELECT ONLY FIXED FEATURES
# ============================================================
X_raw = X_raw[:, FIXED_FEATURES_IDX]
selected_feature_names = [X_df.columns[i] for i in FIXED_FEATURES_IDX]
print("Using fixed features:", selected_feature_names)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_pred),
        "PR-AUC": average_precision_score(y_true, y_pred),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }


# ============================================================
# QBOOST IMPLEMENTATION
# ============================================================
def build_weak_learners(X_train, y_train, n_weak=N_WEAK):
    """Generate a pool of weak learners (decision stumps)."""
    models = []
    preds_matrix = []  # shape: (n_weak, n_train)
    n_samples = len(X_train)

    for _ in range(n_weak):
        stump = DecisionTreeClassifier(max_depth=1)
        idx = np.random.choice(n_samples, n_samples, replace=True)
        stump.fit(X_train[idx], y_train[idx])
        pred = stump.predict(X_train)
        pred = np.where(pred == 0, -1, 1)
        models.append(stump)
        preds_matrix.append(pred)

    preds_matrix = np.array(preds_matrix)
    return models, preds_matrix


def solve_qboost(preds, y_train, lam=LAMBDA, num_reads=NUM_READS_MODEL):
    """
    Encode QBoost selection as a QUBO and solve it.
    Maximize correlation with labels while minimizing interaction and regularize.
    """
    y_spin = np.where(y_train==0, -1, 1)
    M = preds.shape[0]

    # Build QUBO
    Q = {}
    for i in range(M):
        for j in range(M):
            key = (min(i,j), max(i,j))
            # interaction tries to align with label correlations
            interaction = np.sum(preds[i] * preds[j]) / len(y_train)
            Q[key] = Q.get(key, 0) + interaction

    # regularization and correlation with labels
    for i in range(M):
        linear = -2 * np.sum(y_spin * preds[i]) / len(y_train)
        Q[(i,i)] = Q.get((i,i), 0) + linear + lam

    # Sample from QUBO
    sampler = SimulatedAnnealingSampler()
    bqm = dimod.BinaryQuadraticModel.from_qubo(Q)
    sample = sampler.sample(bqm, num_reads=num_reads).first.sample

    # Extract selected weak learners
    w = np.array([sample[i] for i in range(M)])
    return w


def qboost_predict(X_train, y_train, X_test, n_weak=N_WEAK, k_ens=K_ENS):
    models, preds_matrix = build_weak_learners(X_train, y_train, n_weak)
    weights = solve_qboost(preds_matrix, y_train)

    # weighted majority vote
    train_votes = np.zeros(len(X_train))
    test_votes = np.zeros(len(X_test))

    for w, model in zip(weights, models):
        if w == 1:
            p_train = model.predict(X_train)
            p_test = model.predict(X_test)
            p_train = np.where(p_train == 0, -1, 1)
            p_test = np.where(p_test == 0, -1, 1)
            train_votes += p_train
            test_votes += p_test

    # bias to center
    bias = -np.mean(train_votes)

    # final decision
    decision = test_votes + bias
    y_pred = (decision >= 0).astype(int)

    return y_pred


# ============================================================
# MAIN LOOP WITH REPEATED RANDOM SUBSAMPLING
# ============================================================
summary = []
completed_jobs = 0
total_models = 1  # only QBoost

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, n_splits + 1):
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=split * RANDOM_SEED_BASE
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    start = time.time()
    y_pred = qboost_predict(X_train, y_train, X_test)
    runtime = time.time() - start

    metrics = compute_metrics(y_test, y_pred)
    completed_jobs += 1
    progress = (completed_jobs / (total_models * n_splits)) * 100

    print(
        f"[{progress:6.2f}%] "
        f"QBOOST | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split": split,
        "Accuracy_mean": metrics["Accuracy"],
        "ROC_AUC_mean": metrics["ROC-AUC"],
        "F1_mean": metrics["F1"],
        "Precision": metrics["Precision"],
        "Sensitivity": metrics["Sensitivity"],
        "Specificity": metrics["Specificity"],
        "Kappa": metrics["Kappa"],
        "Runtime_sec": runtime
    }
    summary.append(row)
    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)


# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary).sort_values("Accuracy_mean", ascending=False).reset_index(drop=True)
print("\n===== FINAL SORTED RESULTS =====")
display(summary_df)
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>extra**